# GemiDrie: PML + Dataset + Transfer Training

This notebook does three things:
1. Plot and animate PML settings.
2. Build paired datasets for transfer learning (`u_{\omega'}` <-> `u_{\omega}`).
3. Train `T_up` and `T_down` on **solutions** `u`, then test as a GMRES preconditioner block.

Design choices follow Kees' preconditioner idea:
- `T_down`: high frequency -> low frequency.
- Solve low-frequency system with direct solver.
- `T_up`: low frequency -> high frequency.
- Compose this in preconditioning action `M^{-1}`.


## Imports
Load numerical, plotting, and deep-learning dependencies used throughout the notebook.


In [ ]:
from __future__ import annotations

import time
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import scipy.sparse as sp
import scipy.sparse.linalg as spla
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset


## Configuration
Define PML settings, fixed total grid size (`500x500`), solver settings, and training settings.

This notebook uses a **fixed total grid**. The physical interior is `n_phys = n_tot - 2*npml`.


In [ ]:
PML_CONFIG = {
    16:  {"npml": 104, "eta": 70.0},
    32:  {"npml": 88,  "eta": 110.0},
    64:  {"npml": 72,  "eta": 190.0},
    128: {"npml": 60,  "eta": 320.0},
}

PLOT_N_TOT = 500  # plotting grid: total size (physical + PML)

@dataclass
class GridCfg:
    n_tot_target: int = 500
    omega_low: int = 64
    omega_high: int = 128

    @property
    def pml(self):
        return PML_CONFIG[self.omega_high]

    @property
    def n_pml(self) -> int:
        return int(self.pml["npml"])

    @property
    def eta(self) -> float:
        return float(self.pml["eta"])

    @property
    def pml_power(self) -> float:
        return float(self.pml.get("pml_power", 2.0))

    @property
    def n_tot(self) -> int:
        return int(self.n_tot_target)

    @property
    def n_phys(self) -> int:
        nphys = self.n_tot - 2 * self.n_pml
        if nphys <= 2:
            raise ValueError(f"n_phys={nphys} invalid; n_tot={self.n_tot}, npml={self.n_pml}")
        return int(nphys)

    @property
    def h(self) -> float:
        return 1.0 / (self.n_tot - 1)

@dataclass
class RunCfg:
    seed: int = 42
    n_train: int = 128
    n_val: int = 24
    n_test: int = 24
    n_src_min: int = 3
    n_src_max: int = 6
    source_margin: int = 16
    batch_size: int = 1
    epochs: int = 10
    lr: float = 1e-3
    gmres_tol: float = 1e-8
    gmres_maxiter: int = 80
    stencil_order: int = 2

grid_cfg = GridCfg()
run_cfg = RunCfg()

np.random.seed(run_cfg.seed)
torch.manual_seed(run_cfg.seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)
print("grid:", grid_cfg)
print("derived n_phys:", grid_cfg.n_phys)
print("run:", run_cfg)
print("plot total grid:", PLOT_N_TOT)


## PML Visualization Helpers
These functions visualize 1D damping and a 2D damping map on a fixed total grid.


In [ ]:
def _pml_power(cfg: dict[str, float]) -> float:
    return float(cfg.get("pml_power", 2.0))


def sigma_profile_1d(n_tot: int, n_pml: int, eta: float, pml_power: float) -> np.ndarray:
    sig = np.zeros(n_tot, dtype=float)
    if n_pml <= 0:
        return sig

    for i in range(n_tot):
        if i < n_pml:
            dist = (n_pml - i) / n_pml
        elif i >= (n_tot - n_pml):
            dist = (i - (n_tot - n_pml) + 1) / n_pml
        else:
            dist = 0.0
        sig[i] = eta * (dist ** pml_power)
    return sig


def sigma_map_2d(n_tot: int, n_pml: int, eta: float, pml_power: float) -> np.ndarray:
    sx = sigma_profile_1d(n_tot, n_pml, eta, pml_power)
    return sx[:, None] + sx[None, :]


def plot_sigma_grid(pml_config: dict[int, dict[str, float]], n_tot_plot: int = 500):
    fig, axs = plt.subplots(2, 2, figsize=(12, 8), constrained_layout=True)
    axs = axs.ravel()

    for ax, (om, cfg) in zip(axs, sorted(pml_config.items())):
        npml = int(cfg["npml"])
        eta = float(cfg["eta"])
        pml_power = _pml_power(cfg)
        n_tot = int(n_tot_plot)
        n_phys = n_tot - 2 * npml
        if n_phys <= 0:
            ax.set_title(f"ω={om} invalid: n_tot={n_tot} too small for npml={npml}")
            ax.axis("off")
            continue

        sig = sigma_profile_1d(n_tot, npml, eta, pml_power)
        ax.plot(sig, lw=2)
        ax.axvline(npml, color="k", ls="--", lw=1)
        ax.axvline(n_tot - npml - 1, color="k", ls="--", lw=1)
        ax.set_title(f"ω={om} | n_tot={n_tot}, n_phys={n_phys}, npml={npml}, η={eta}, p={pml_power}")
        ax.set_xlabel("grid index i")
        ax.set_ylabel("σ(i)")
        ax.grid(alpha=0.3)

    plt.show()


def plot_sigma_map(*, omega: int, pml_cfg: dict[str, float], n_tot_plot: int = 500):
    npml = int(pml_cfg["npml"])
    eta = float(pml_cfg["eta"])
    pml_power = _pml_power(pml_cfg)
    n_tot = int(n_tot_plot)
    n_phys = n_tot - 2 * npml
    if n_phys <= 0:
        raise ValueError(f"Invalid geometry: n_tot={n_tot}, npml={npml}")

    smap = sigma_map_2d(n_tot, npml, eta, pml_power)
    fig, ax = plt.subplots(figsize=(6.5, 5.5))
    im = ax.imshow(smap, origin="lower", cmap="viridis")
    ax.axhline(npml, color="w", ls="--", lw=1)
    ax.axhline(n_tot - npml - 1, color="w", ls="--", lw=1)
    ax.axvline(npml, color="w", ls="--", lw=1)
    ax.axvline(n_tot - npml - 1, color="w", ls="--", lw=1)
    ax.set_title(f"2D PML damping map (ω={omega}) on fixed {n_tot}x{n_tot} grid (n_phys={n_phys})")
    ax.set_xlabel("x-index")
    ax.set_ylabel("y-index")
    plt.colorbar(im, ax=ax, label="σx+σy")
    plt.show()


def animate_sigma_over_time(*, omega: int, pml_cfg: dict[str, float], n_tot_plot: int = 500, n_frames: int = 40, save_path: str | None = None):
    npml = int(pml_cfg["npml"])
    eta = float(pml_cfg["eta"])
    pml_power = _pml_power(pml_cfg)
    n_tot = int(n_tot_plot)
    n_phys = n_tot - 2 * npml
    if n_phys <= 0:
        raise ValueError(f"Invalid geometry: n_tot={n_tot}, npml={npml}")

    base_1d = sigma_profile_1d(n_tot, npml, eta, pml_power)
    base_2d = sigma_map_2d(n_tot, npml, eta, pml_power)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5), constrained_layout=True)
    line, = ax1.plot(np.zeros_like(base_1d), lw=2)
    ax1.axvline(npml, color="k", ls="--", lw=1)
    ax1.axvline(n_tot - npml - 1, color="k", ls="--", lw=1)
    ax1.set_ylim(0.0, max(1e-12, base_1d.max() * 1.05))
    ax1.set_xlabel("grid index i")
    ax1.set_ylabel("σ_t(i)")
    ax1.grid(alpha=0.3)

    im = ax2.imshow(np.zeros_like(base_2d), origin="lower", cmap="viridis", vmin=0.0, vmax=max(1e-12, base_2d.max()))
    ax2.axhline(npml, color="w", ls="--", lw=1)
    ax2.axhline(n_tot - npml - 1, color="w", ls="--", lw=1)
    ax2.axvline(npml, color="w", ls="--", lw=1)
    ax2.axvline(n_tot - npml - 1, color="w", ls="--", lw=1)
    ax2.set_xlabel("x-index")
    ax2.set_ylabel("y-index")
    plt.colorbar(im, ax=ax2, label="σx+σy")

    def update(k):
        t = k / (n_frames - 1)
        line.set_ydata(t * base_1d)
        im.set_data(t * base_2d)
        ax1.set_title(f"1D σ profile (ω={omega}, t={t:.2f})")
        ax2.set_title(f"2D damping map (n_tot={n_tot}, n_phys={n_phys}, t={t:.2f})")
        return (line, im)

    ani = FuncAnimation(fig, update, frames=n_frames, interval=120, blit=False)
    if save_path is not None:
        out = Path(save_path)
        out.parent.mkdir(parents=True, exist_ok=True)
        ani.save(out, dpi=120)
        print(f"saved animation: {out}")
    plt.show()
    return ani


## Visualize PML Setup
Inspect 1D and 2D PML damping and generate a time-ramp animation.


In [ ]:
plot_sigma_grid(PML_CONFIG, n_tot_plot=PLOT_N_TOT)
plot_sigma_map(
    omega=grid_cfg.omega_high,
    pml_cfg=PML_CONFIG[grid_cfg.omega_high],
    n_tot_plot=PLOT_N_TOT,
)

_ani = animate_sigma_over_time(
    omega=grid_cfg.omega_high,
    pml_cfg=PML_CONFIG[grid_cfg.omega_high],
    n_tot_plot=PLOT_N_TOT,
    n_frames=50,
    save_path="experiments/plots/pml_ramp_omega_high.mp4",
)


## Helmholtz Operator Assembly
Single operator path used in this notebook:
- fixed total grid `n_tot`
- PML is a collar inside that grid
- optional 2nd or 4th order finite-difference stencil


In [ ]:
def get_helmholtz_matrix(
    *,
    omega: float,
    n_tot: int,
    n_pml: int,
    eta: float,
    pml_power: float = 2.0,
    stencil_order: int = 2,
):
    if stencil_order not in (2, 4):
        raise ValueError(f"stencil_order must be 2 or 4, got {stencil_order}")
    if n_tot < 5:
        raise ValueError(f"n_tot must be >= 5, got {n_tot}")
    if n_pml < 0:
        raise ValueError(f"n_pml must be >= 0, got {n_pml}")
    if 2 * n_pml >= n_tot - 2:
        raise ValueError(f"PML too thick: n_tot={n_tot}, n_pml={n_pml}")

    h = 1.0 / (n_tot - 1)

    sig = sigma_profile_1d(n_tot, n_pml, eta, pml_power)
    s = 1.0 / (1.0 + 1j * sig / (omega / (2.0 * np.pi)))

    rows, cols, vals = [], [], []

    def add(r, c, v):
        rows.append(r)
        cols.append(c)
        vals.append(v)

    def lin(i, j):
        return i * n_tot + j

    d2, o21 = -2.0, 1.0
    d4, o41, o42 = -2.5, 4.0 / 3.0, -1.0 / 12.0

    k2 = float(omega) * float(omega)

    for i in range(n_tot):
        sy2 = s[i] * s[i]
        for j in range(n_tot):
            r = lin(i, j)

            if i == 0 or j == 0 or i == n_tot - 1 or j == n_tot - 1:
                add(r, r, 1.0 + 0.0j)
                continue

            sx2 = s[j] * s[j]
            use_2nd = (
                stencil_order == 2
                or i < 2 or j < 2
                or i > n_tot - 3 or j > n_tot - 3
            )

            if use_2nd:
                add(r, r, ((sx2 * d2 + sy2 * d2) / (h * h)) + k2)
                add(r, lin(i, j - 1), sx2 * o21 / (h * h))
                add(r, lin(i, j + 1), sx2 * o21 / (h * h))
                add(r, lin(i - 1, j), sy2 * o21 / (h * h))
                add(r, lin(i + 1, j), sy2 * o21 / (h * h))
            else:
                add(r, r, ((sx2 * d4 + sy2 * d4) / (h * h)) + k2)
                add(r, lin(i, j - 1), sx2 * o41 / (h * h))
                add(r, lin(i, j + 1), sx2 * o41 / (h * h))
                add(r, lin(i, j - 2), sx2 * o42 / (h * h))
                add(r, lin(i, j + 2), sx2 * o42 / (h * h))
                add(r, lin(i - 1, j), sy2 * o41 / (h * h))
                add(r, lin(i + 1, j), sy2 * o41 / (h * h))
                add(r, lin(i - 2, j), sy2 * o42 / (h * h))
                add(r, lin(i + 2, j), sy2 * o42 / (h * h))

    A = sp.coo_matrix((vals, (rows, cols)), shape=(n_tot * n_tot, n_tot * n_tot)).tocsr()
    return A


## Dataset Builder
Generate paired training targets on the same grid:
- `T_up`: `u_low -> u_high`
- `T_down`: `u_high -> u_low`

Right-hand side sources are randomized in non-PML interior.


In [ ]:
def make_rhs(n_tot: int, n_pml: int, rng: np.random.Generator, n_sources: int, margin: int):
    if n_sources < 1:
        raise ValueError("n_sources must be >= 1")

    f = np.zeros((n_tot, n_tot), dtype=np.complex128)
    lo = n_pml + margin
    hi = n_tot - n_pml - margin
    if hi <= lo:
        raise ValueError(
            f"source margin too large: n_tot={n_tot}, n_pml={n_pml}, margin={margin}"
        )

    for _ in range(n_sources):
        y = int(rng.integers(lo, hi))
        x = int(rng.integers(lo, hi))
        amp = float(rng.uniform(1.0, 2.0))
        phase = float(rng.uniform(0.0, 2 * np.pi))
        f[y, x] += amp * np.exp(1j * phase)
    return f


def to_2ch(u: np.ndarray) -> np.ndarray:
    return np.stack([u.real, u.imag], axis=0).astype(np.float32)


def build_transfer_dataset(grid_cfg: GridCfg, run_cfg: RunCfg, n_samples: int, seed_offset: int = 0):
    rng = np.random.default_rng(run_cfg.seed + seed_offset)

    A_low = get_helmholtz_matrix(
        omega=float(grid_cfg.omega_low),
        n_tot=grid_cfg.n_tot,
        n_pml=grid_cfg.n_pml,
        eta=grid_cfg.eta,
        pml_power=grid_cfg.pml_power,
        stencil_order=run_cfg.stencil_order,
    )
    A_high = get_helmholtz_matrix(
        omega=float(grid_cfg.omega_high),
        n_tot=grid_cfg.n_tot,
        n_pml=grid_cfg.n_pml,
        eta=grid_cfg.eta,
        pml_power=grid_cfg.pml_power,
        stencil_order=run_cfg.stencil_order,
    )

    solve_low = spla.factorized(A_low.tocsc())
    solve_high = spla.factorized(A_high.tocsc())

    X_up, Y_up = [], []
    X_down, Y_down = [], []
    rhs_list = []

    n_tot = grid_cfg.n_tot

    for _ in tqdm(range(n_samples), desc="Building dataset", leave=False):
        nsrc = int(rng.integers(run_cfg.n_src_min, run_cfg.n_src_max + 1))
        rhs = make_rhs(n_tot, grid_cfg.n_pml, rng, nsrc, run_cfg.source_margin)
        b = rhs.reshape(-1)

        u_low = solve_low(b).reshape(n_tot, n_tot)
        u_high = solve_high(b).reshape(n_tot, n_tot)

        X_up.append(to_2ch(u_low))
        Y_up.append(to_2ch(u_high))

        X_down.append(to_2ch(u_high))
        Y_down.append(to_2ch(u_low))

        rhs_list.append(rhs)

    return {
        "X_up": np.stack(X_up, axis=0),
        "Y_up": np.stack(Y_up, axis=0),
        "X_down": np.stack(X_down, axis=0),
        "Y_down": np.stack(Y_down, axis=0),
        "rhs": rhs_list,
        "A_low": A_low,
        "A_high": A_high,
        "solve_low": solve_low,
        "solve_high": solve_high,
    }


## Build Train/Validation/Test Splits
Build all splits with timing so progress is visible.


In [ ]:
splits = [
    ("train", run_cfg.n_train, 0),
    ("val", run_cfg.n_val, 1000),
    ("test", run_cfg.n_test, 2000),
]

all_data = {}
t0_all = time.perf_counter()

for name, n_samples, seed_off in splits:
    t0 = time.perf_counter()
    print(f"[{name}] start: {n_samples} samples")
    all_data[name] = build_transfer_dataset(grid_cfg, run_cfg, n_samples, seed_offset=seed_off)
    dt = time.perf_counter() - t0
    print(f"[{name}] done in {dt:.1f}s ({n_samples / max(dt,1e-9):.2f} samples/s)")

train_data = all_data["train"]
val_data = all_data["val"]
test_data = all_data["test"]

print(f"total dataset build time: {time.perf_counter() - t0_all:.1f}s")
print("train X_up:", train_data["X_up"].shape)
print("train X_down:", train_data["X_down"].shape)
print("A_high shape:", train_data["A_high"].shape)


## Transfer Models and Training Utilities
Use a shallow CNN baseline first (`width=16`, `dilation=1`) with per-epoch and per-batch progress.


In [ ]:
class TransferCNN(nn.Module):
    def __init__(self, width: int = 16, dilation: int = 1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(2, width, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(width, width, 3, padding=dilation, dilation=dilation),
            nn.ReLU(inplace=True),
            nn.Conv2d(width, 2, 3, padding=1),
        )

    def forward(self, x):
        return self.net(x)


def train_model(model: nn.Module, X: np.ndarray, Y: np.ndarray, run_cfg: RunCfg, Xv: np.ndarray, Yv: np.ndarray, tag: str = "T"):
    model = model.to(device)

    X_t = torch.tensor(X, dtype=torch.float32)
    Y_t = torch.tensor(Y, dtype=torch.float32)
    ds = TensorDataset(X_t, Y_t)
    dl = DataLoader(ds, batch_size=run_cfg.batch_size, shuffle=True, pin_memory=(device.type == "cuda"))

    xv = torch.tensor(Xv, dtype=torch.float32, device=device)
    yv = torch.tensor(Yv, dtype=torch.float32, device=device)

    opt = optim.Adam(model.parameters(), lr=run_cfg.lr)
    loss_fn = nn.MSELoss()

    hist = []
    for ep in range(run_cfg.epochs):
        model.train()
        total = 0.0
        pbar = tqdm(dl, desc=f"{tag} epoch {ep+1}/{run_cfg.epochs}", leave=False)

        for xb, yb in pbar:
            xb = xb.to(device, non_blocking=True)
            yb = yb.to(device, non_blocking=True)

            opt.zero_grad(set_to_none=True)
            pred = model(xb)
            loss = loss_fn(pred, yb)
            loss.backward()
            opt.step()

            l = float(loss.item())
            total += l
            pbar.set_postfix(batch_mse=f"{l:.3e}")

        model.eval()
        with torch.no_grad():
            val_loss = float(loss_fn(model(xv), yv).item())

        tr_loss = total / max(1, len(dl))
        hist.append({"epoch": ep + 1, "train_mse": tr_loss, "val_mse": val_loss})
        print(f"{tag} epoch={ep+1:03d}/{run_cfg.epochs} train_mse={tr_loss:.4e} val_mse={val_loss:.4e}")

    return model, hist


@torch.no_grad()
def eval_mse(model: nn.Module, X: np.ndarray, Y: np.ndarray) -> float:
    model.eval()
    x = torch.tensor(X, dtype=torch.float32, device=device)
    y = torch.tensor(Y, dtype=torch.float32, device=device)
    pred = model(x)
    return float(torch.mean((pred - y) ** 2).item())


## Train `T_up` and `T_down`
Train on solution pairs, then report validation/test MSE.


In [ ]:
T_up = TransferCNN(width=16, dilation=1)
T_up, hist_up = train_model(
    T_up,
    train_data["X_up"],
    train_data["Y_up"],
    run_cfg,
    val_data["X_up"],
    val_data["Y_up"],
    tag="T_up",
)

T_down = TransferCNN(width=16, dilation=1)
T_down, hist_down = train_model(
    T_down,
    train_data["X_down"],
    train_data["Y_down"],
    run_cfg,
    val_data["X_down"],
    val_data["Y_down"],
    tag="T_down",
)

metrics = {
    "T_up_val_mse": eval_mse(T_up, val_data["X_up"], val_data["Y_up"]),
    "T_down_val_mse": eval_mse(T_down, val_data["X_down"], val_data["Y_down"]),
    "T_up_test_mse": eval_mse(T_up, test_data["X_up"], test_data["Y_up"]),
    "T_down_test_mse": eval_mse(T_down, test_data["X_down"], test_data["Y_down"]),
}
metrics


## Save Trained Operators
Persist checkpoints for later GMRES/FGMRES experiments.


In [ ]:
out = Path("experiments/checkpoints")
out.mkdir(parents=True, exist_ok=True)
torch.save(T_up.state_dict(), out / "T_up_gemidrie.pth")
torch.save(T_down.state_dict(), out / "T_down_gemidrie.pth")
print("saved to", out)


## Preconditioner Block for GMRES / FGMRES
Compose learned transfer with low-frequency direct solve:
1. `w_L = T_down(v_H)`
2. `z_L = A_L^{-1} w_L`
3. `z_H = T_up(z_L)`

For nonlinear operators, prefer FGMRES.


In [ ]:
@torch.no_grad()
def apply_net_to_vec(net: nn.Module, v: np.ndarray, n_tot: int) -> np.ndarray:
    net.eval()
    u = v.reshape(n_tot, n_tot)
    x = torch.tensor(to_2ch(u)[None, ...], dtype=torch.float32, device=device)
    y = net(x).detach().cpu().numpy()[0]
    out = y[0] + 1j * y[1]
    return out.reshape(-1)


@torch.no_grad()
def make_M_apply(T_down: nn.Module, T_up: nn.Module, solve_low, n_tot: int):
    def M_apply(v_high: np.ndarray) -> np.ndarray:
        w_low = apply_net_to_vec(T_down, v_high, n_tot=n_tot)
        z_low = solve_low(w_low)
        z_high = apply_net_to_vec(T_up, z_low, n_tot=n_tot)
        return z_high
    return M_apply


def run_gmres(A: sp.csr_matrix, rhs: np.ndarray, tol: float, maxiter: int, M_apply=None):
    b = rhs.reshape(-1)
    hist = []

    def cb(res):
        hist.append(float(res))

    kwargs = dict(atol=tol, maxiter=maxiter, restart=None, callback=cb, callback_type="legacy")
    if M_apply is None:
        _, info = spla.gmres(A, b, **kwargs)
    else:
        n2 = b.size
        M = spla.LinearOperator((n2, n2), matvec=M_apply, dtype=np.complex128)
        _, info = spla.gmres(A, b, M=M, **kwargs)

    return {
        "info": int(info),
        "iters": int(len(hist)),
        "final_res": float(hist[-1] if hist else np.inf),
        "hist": hist,
    }


## Compare GMRES vs ML-Preconditioned GMRES
Evaluate residual reduction on test right-hand sides.


In [ ]:
A_high = test_data["A_high"]
solve_low = test_data["solve_low"]
M_apply = make_M_apply(T_down, T_up, solve_low, n_tot=grid_cfg.n_tot)

rows = []
for i, rhs in enumerate(test_data["rhs"]):
    base = run_gmres(A_high, rhs, run_cfg.gmres_tol, run_cfg.gmres_maxiter, M_apply=None)
    mlpc = run_gmres(A_high, rhs, run_cfg.gmres_tol, run_cfg.gmres_maxiter, M_apply=M_apply)
    rows.append({"sample": i, "base": base, "mlpc": mlpc})

avg_base = float(np.mean([r["base"]["final_res"] for r in rows]))
avg_mlpc = float(np.mean([r["mlpc"]["final_res"] for r in rows]))

print("avg final residual (no preconditioner):", avg_base)
print("avg final residual (ML preconditioner):", avg_mlpc)
rows[0]


## Notes
This notebook now keeps a single consistent path:
- fixed total grid (`n_tot=500`) with PML inside
- one Helmholtz assembly function
- one dataset builder
- one shallow baseline model/training loop
- one GMRES evaluation path
